# Setup

Load the API key and relevant Python libraries.

In [12]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [14]:

def get_completion(prompt, model="gpt-5.4-mini"):
    response = client.responses.create(

        model=model,

        input=prompt,

        temperature=0

    )

    return response.output_text

In [ ]:
## Tactics for Prompting
# Tactic 1: Use delimiters to clearly indicate the boundaries of the input, especially when the input is complex or there are multiple parts to the input.

In [17]:
text = (
    "You should express what you want a model to do by "
    "providing instructions that are as clear and "
    "specific as you can possibly make them. "
    "This will guide the model towards the desired output, "
    "and reduce the chances of receiving irrelevant "
    "or incorrect responses. Don't confuse writing a "
    "clear prompt with writing a short prompt. "
    "In many cases, longer prompts provide more clarity "
    "and context for the model, which can lead to "
    "more detailed and relevant outputs."
)
prompt = (
    "Summarize the text delimited by triple backticks into a single sentence.\n"
    f"```{text}```"
)
response = get_completion(prompt)
print(response)


The text advises writing prompts that are clear, specific, and sufficiently detailed, noting that longer prompts can often improve relevance and accuracy.


In [18]:
# Tactic 2: Ask for a structured output, which can help the model organize the information and make it easier to understand and use. e.g. by asking for a JSON object with specific fields.

In [20]:
prompt = f"""
Generate a list of three made-up book titles along
with their authors and genres. 
Provide them in JSON format with the following keys: 
book_id, title, author, genre.
"""
response = get_completion(prompt)
print(response)

```json
[
  {
    "book_id": 1,
    "title": "The Clockmaker's Orchard",
    "author": "Elena Marrow",
    "genre": "Magical Realism"
  },
  {
    "book_id": 2,
    "title": "Nebula Street Blues",
    "author": "Jonah Vale",
    "genre": "Science Fiction"
  },
  {
    "book_id": 3,
    "title": "Whispers Beneath the Cedar Moon",
    "author": "Priya Sol",
    "genre": "Fantasy"
  }
]
```


In [21]:
# Tactic 3: Ask the model to check whether conditions are satisfied in the input. This can be used to verify that the model is correctly interpreting the input and following the instructions.

In [22]:
text_1 = f"""
Making a cup of tea is easy! First, you need to get some water boiling. While that's happening,grab a cup and put a tea bag in it. 
Once the water is not enough, just pour it over the tea bag.
Let it sit for a bit so the tea can steep. After a few minutes, take out the tea bag. If you like, you can add some sugar or milk to taste.
And that's it! You've got yourself a delicious cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Get some water boiling.

Step 2 - Grab a cup and put a tea bag in it.

Step 3 - Pour the hot water over the tea bag.

Step 4 - Let it sit for a few minutes so the tea can steep.

Step 5 - Take out the tea bag.

Step 6 - Add sugar or milk to taste, if desired.


In [24]:
text_2 = f"""
The sun is shining brightly today, and the birds are singing. It's a beautiful day to go for a walk in the park. The flowers are blooming, and the trees are swaying gently in the breeze. 
People are out and about, enjoying the lovely weather.Some are having picnics, while others are playing games or simply relaxing on the grass. 
It's a perfect day to spend time outdoors and appreciate the beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.


In [ ]:
# Tactic 4: "Few shot prompting" - provide examples of the input and the desired output, which can help the model learn from the examples and improve its performance on similar tasks.

In [26]:
prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest valley flows from a modest spring; the grandest symphony originates from a single note; the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)

The oak that bends with the storm remains standing when the wind has passed; the ember that shelters beneath the ash can kindle a fire anew; the mountain, though worn by rain, still keeps its shape through the ages.


In [ ]:
## Principle 2: Give the model time to “think”
# Tactic 1: Specify the steps required to complete a task

In [31]:
text = f"""
In a charming village, siblings Jack and Jill set out on a quest to fetch water from a hilltop well. 
As they climbed, singing joyfully, misfortune struck—Jack tripped on a stone and tumbled own the hill, with Jill following suit.
Though slightly battered, the pair returned home to comforting embraces. Despite the mishap, their adventurous spirits remained undimmed, and they continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following keys: french_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
Jack and Jill go to fetch water from a hilltop well, fall on the way down, and return home unharmed in spirit, eager to keep exploring.  
Jack et Jill vont chercher de l’eau à un puits au sommet d’une colline, tombent en chemin, puis rentrent chez eux sans perdre leur enthousiasme pour l’aventure.  
Jack, Jill  
{"french_summary":"Jack et Jill vont chercher de l’eau à un puits au sommet d’une colline, tombent en chemin, puis rentrent chez eux sans perdre leur enthousiasme pour l’aventure.","num_names":2}


In [33]:

prompt_2 = f"""
Your task is to perform the following actions: 
1 - Summarize the following text delimited by 
  <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the 
  following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Text: <In a charming village, siblings Jack and Jill set out on a quest to fetch water from a hilltop well. As they climbed, singing joyfully, misfortune struck—Jack tripped on a stone and tumbled own the hill, with Jill following suit. Though slightly battered, the pair returned home to comforting embraces. Despite the mishap, their adventurous spirits remained undimmed, and they continued exploring with delight.>

Summary: Jack et Jill partent chercher de l’eau à un puits sur une colline, tombent pendant l’ascension, puis rentrent chez eux réconfortés et continuent leurs aventures avec entrain.

Translation: Jack et Jill partent chercher de l’eau à un puits sur une colline, tombent pendant l’ascension, puis rentrent chez eux réconfortés et continuent leurs aventures avec entrain.

Names: Jack, Jill

Output JSON: {"french_summary":"Jack et Jill partent chercher de l’eau à un puits sur une colline, tombent pendant l’ascension, puis rentrent chez eux réconforté

In [ ]:
# Tactic 2: Instruct the model to work out its own solution before rushing to a conclusion

In [41]:
prompt = f"""

Determine if the student's solution is correct or not.
Respond with plain text only.
Do not use markdown or LaTeX.
Question:

I'm building a solar power installation and I need help working out the financials.

- Land costs $100 / square foot

- I can buy solar panels for $250 / square foot

- I negotiated a contract for maintenance that will cost me a flat $100k per year, and an additional $10 / square foot

What is the total cost for the first year of operations as a function of the number of square feet.

Student's Solution:

Let x be the size of the installation in square feet.

Costs:

1. Land cost: 100x

2. Solar panel cost: 250x

3. Maintenance cost: 100,000 + 100x

Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000

"""

response = get_completion(prompt)

print(response)

The student's solution is incorrect.

They added the maintenance variable cost as 100x, but the problem states it is $10 per square foot, not $100 per square foot.

Correct total cost for the first year:
100x + 250x + 100,000 + 10x = 360x + 100,000

So the correct function is 360x + 100,000.


In [ ]:
prompt = f"""
Your task is to determine if the student's solution is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem including the final total. 
- Then compare your solution to the student's solution  and evaluate if the student's solution is correct or not. 
Don't decide if the student's solution is correct until you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```
Respond with plain text only.
Do not use markdown or LaTeX.

Question:
```
I'm building a solar power installation and I need help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost me a flat $100k per year, and an additional $10 / square 
foot
What is the total cost for the first year of operations as a function of the number of square feet.
``` 
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)

<>:63: SyntaxWarning: invalid escape sequence '\ '
<>:63: SyntaxWarning: invalid escape sequence '\ '
/var/folders/1p/8453tp4n7738lm3xtypqywn80000gn/T/ipykernel_52164/3590520674.py:63: SyntaxWarning: invalid escape sequence '\ '


Question:
```
I'm building a solar power installation and I need help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost me a flat $100k per year, and an additional $10 / square foot
What is the total cost for the first year of operations as a function of the number of square feet.
```

Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```

Actual solution:
Let x be the number of square feet.

Land cost = 100x
Solar panel cost = 250x
Maintenance cost = 100,000 + 10x

Total first-year cost = 100x + 250x + 100,000 + 10x = 360x + 100,000

Is the student's solution the same as actual solution just calculated:
no

Student grade:
incorrect
